Load csv dataset and drop low minute players

In [5]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

MIN_MINUTES = 900 #10 full matches

plyrs24_25 = pd.read_csv("../data/raw/players_data-2024_2025.csv")
plyrs25_26 = pd.read_csv("../data/raw/players_data-2025_2026.csv")

plyrs24_25 = plyrs24_25[plyrs24_25['Min'] >= MIN_MINUTES]
plyrs25_26 = plyrs25_26[plyrs25_26['Min'] >= MIN_MINUTES]

# Unneccesary features that make things bloated
DROP_FEATURES = ['Rk_stats_keeper', 'Nation_stats_keeper', 'Pos_stats_keeper', 'Comp_stats_keeper', 'Age_stats_keeper',
                 'Born_stats_keeper', 'MP_stats_keeper', 'Starts_stats_keeper', 'Min_stats_keeper', '90s_stats_keeper', 
                'PK_stats_shooting', 'PKatt_stats_shooting', 'Rk_stats_playing_time', 'Nation_stats_playing_time', 'Pos_stats_playing_time', 
                'Comp_stats_playing_time', 'Age_stats_playing_time', 'Born_stats_playing_time', 'MP_stats_playing_time', 'Min_stats_playing_time',
                 'Rk_stats_shooting', 'Nation_stats_shooting', 'Pos_stats_shooting', 'Comp_stats_shooting', 'Age_stats_shooting', 'Born_stats_shooting',
                 '90s_stats_shooting', 'Gls_stats_shooting', 'Rk_stats_misc', 'Nation_stats_misc', 'Pos_stats_misc', 'Comp_stats_misc', 'Age_stats_misc', 
                'Born_stats_misc', '90s_stats_misc', 'CrdY_stats_misc', 'CrdR_stats_misc', '90s_stats_playing_time', 'Starts_stats_playing_time', 'Nation_stats_passing', 
                 'Pos_stats_passing', 'Comp_stats_passing', 'Nation_stats_passing_types', 'Pos_stats_passing_types', 'Comp_stats_passing_types', 'Nation_stats_gca', 
                 'Pos_stats_gca', 'Comp_stats_gca', 'Nation_stats_defense', 'Pos_stats_defense', 'Comp_stats_defense', 'Nation_stats_possession', 'Pos_stats_possession', 
                 'Comp_stats_possession', 'Nation_stats_keeper_adv', 'Pos_stats_keeper_adv', 'Comp_stats_keeper_adv'
                ]

plyrs24_25 = plyrs24_25.drop(columns=DROP_FEATURES)
plyrs25_26 = plyrs25_26.drop(columns=DROP_FEATURES)

plyrs24_25 = plyrs24_25.set_index('Player')
plyrs25_26 = plyrs25_26.set_index('Player')

metadata = ['Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born']

features = [col for col in plyrs24_25.columns if col not in metadata]

WEIGHT_24_25 = 0.3
WEIGHT_25_26 = 0.7

features_24_25 = plyrs24_25[features]
features_25_26 = plyrs25_26[features]

features_24_25, features_25_26 = features_24_25.align(features_25_26, join = "outer", axis=0)

features_24_25 = features_24_25.select_dtypes(include="number")
features_25_26 = features_25_26.select_dtypes(include="number")

combined_features = (WEIGHT_24_25 * features_24_25.fillna(0)) + (WEIGHT_25_26 * features_25_26.fillna(0))

players = combined_features.join(plyrs25_26[metadata], how = 'inner')

players = players.copy() # get rid of fragmented df warning, avoids bad performence
players = players.reset_index()

players['PlayerID'] = range(1, len(players) + 1)
players = players.set_index('PlayerID')

players.head()

,Player,#OPA,#OPA/90,+/-,+/-90,/90,1/3,1/3_stats_possession,2CrdY,90s,...,xG+/-,xG+/-90,xG+xAG,xG_stats_shooting,Nation,Pos,Squad,Comp,Age,Born
PlayerID,,,,,,,,,,,,,,,,,,,,,
1,Aaron Wan-Bissaka,0.0,0.000,-12.9,-0.878,0.00,46.4,29.3,0.0,18.34,...,-10.05,-0.645,0.084,0.36,cd COD,DF,West Ham,eng Premier League,28.0,1997.0
2,Aaron Zehnter,0.0,0.000,-0.7,-0.063,0.00,12.6,7.7,0.0,7.70,...,-3.08,-0.280,0.098,0.35,de GER,DF,Wolfsburg,de Bundesliga,21.0,2004.0
3,Aarón Escandell,25.9,1.365,-13.3,-0.700,0.28,4.2,0.0,0.0,13.30,...,-10.57,-0.560,0.000,0.00,es ESP,GK,Oviedo,es La Liga,30.0,1995.0
4,Aarón Martín,0.0,0.000,-7.5,-0.390,0.00,51.1,17.0,0.0,20.37,...,-2.92,-0.115,0.337,0.48,es ESP,DF,Genoa,it Serie A,28.0,1997.0
5,Abdoul Coulibaly,0.0,0.000,-5.6,-0.448,0.00,25.9,2.8,0.7,8.82,...,-3.92,-0.308,0.035,0.35,de GER,DF,Werder Bremen,de Bundesliga,18.0,2007.0


In [12]:
players.isna().sum()

Player     0
#OPA       0
#OPA/90    0
+/-        0
+/-90      0
          ..
Pos        0
Squad      0
Comp       0
Age        0
Born       0
Length: 210, dtype: int64

In [14]:
for col in players.columns:
    print(col + " ", end="")

Player #OPA #OPA/90 +/- +/-90 /90 1/3 1/3_stats_possession 2CrdY 90s 90s_stats_defense 90s_stats_gca 90s_stats_keeper_adv 90s_stats_passing 90s_stats_passing_types 90s_stats_possession A-xAG Age_stats_defense Age_stats_gca Age_stats_keeper_adv Age_stats_passing Age_stats_passing_types Age_stats_possession Ast Ast_stats_passing Att Att (GK) Att 3rd Att 3rd_stats_possession Att Pen Att_stats_defense Att_stats_keeper_adv Att_stats_passing_types Att_stats_possession AvgDist AvgLen Blocks Blocks_stats_defense Born_stats_defense Born_stats_gca Born_stats_keeper_adv Born_stats_passing Born_stats_passing_types Born_stats_possession CK CK_stats_keeper_adv CPA CS CS% Carries Clr Cmp Cmp% Cmp%_stats_keeper_adv Cmp_stats_keeper_adv Cmp_stats_passing_types Compl CrdR CrdY Crs CrsPA Crs_stats_misc D Dead Def Def 3rd Def 3rd_stats_possession Def Pen Dis Dist Err FK FK_stats_keeper_adv FK_stats_passing_types Fld Fld_stats_misc Fls G+A G+A-PK G-PK G-xG G/Sh G/SoT GA GA90 GA_stats_keeper_adv GCA GCA90 G

In [24]:
players['Comp'].unique().tolist()

['eng Premier League',
 'de Bundesliga',
 'es La Liga',
 'it Serie A',
 'fr Ligue 1']

In [17]:
OUTFIELD_FEATURES = ['90s', 'Gls', 'Ast', 'xG', 'xAG', 'npxG', 'G-PK', 
                     'Tkl', 'TklW','Blocks', 'Int', 'Clr', 'Err',
                     'PrgP', 'PrgC', 'KP', 'PPA', 
                     'Touches', 'Carries', 'PrgR', 'Mis', 'Dis']

In [52]:
def build_league_outfield_df(df, league_name):
    league_df = df[df['Comp'] == league_name]
    league_outfield = league_df[OUTFIELD_FEATURES]
    
    #adjust to a per 90 basis
    league_outfield = league_outfield.apply(lambda x : x / league_outfield['90s'])
    league_outfield = league_outfield.drop(columns='90s')

    #rename columns to reflect per 90 scale
    new_cols = {col:f'{col}/90' for col in league_outfield.columns.tolist()}
    league_outfield = league_outfield.rename(columns=new_cols)

    #Normalize using standard scaler
    scaler = StandardScaler()
    league_outfield = pd.DataFrame(scaler.fit_transform(league_outfield),
                                   columns=league_outfield.columns,
                                   index=league_outfield.index)
    
    return league_outfield

In [80]:
prem_outfield = build_league_outfield_df(players, 'eng Premier League')
laliga_outfield = build_league_outfield_df(players, 'es La Liga')
bundesliga_outfield = build_league_outfield_df(players, 'de Bundesliga')
serieA_outfield = build_league_outfield_df(players, 'it Serie A')
ligue1_outfield = build_league_outfield_df(players, 'fr Ligue 1')

outfield_vectors = pd.concat([prem_outfield, laliga_outfield, bundesliga_outfield, ligue1_outfield])

outfield_vectors.sort_values(by='xG/90', ascending=False).head()
players.loc[385]

Player     Joaquín Panichelli
#OPA                      0.0
#OPA/90                   0.0
+/-                       2.8
+/-90                   0.182
                  ...        
Pos                     FW,MF
Squad              Strasbourg
Comp               fr Ligue 1
Age                      23.0
Born                   2002.0
Name: 385, Length: 210, dtype: object

In [26]:
GOAL_KEEPER_FEATURES = ['GA', 'Saves', 'Save%', 'CS', 'CS%', 'PKsv', '90s']
OUTFIELD_FEATURES = ['Gls', 'Ast', 'Sh', 'SoT/90', 'G/Sh', 'G/SoT', 'Fls', 'Fld', 'Off', 'Int','TklW','90s']
QUALITATIVE_FEATURES = ['Player','Pos', 'Nation', 'Squad', 'Comp', 'Born']

In [27]:
prem = players[players['Comp'] == 'eng Premier League']

In [28]:
prem_outfield = prem[prem['Pos'] != 'GK']
prem_outfield = prem[OUTFIELD_FEATURES]

#Missing values found in G/Sh and G/SoT, fill with 0
prem_outfield['G/Sh'] = prem_outfield['G/Sh'].fillna(0)
prem_outfield['G/SoT'] = prem_outfield['G/SoT'].fillna(0)

#Adjusting Stats on a Per 90 Basis
cols_to_change = ['Gls', 'Ast', 'Sh', 'Fls', 'Fld', 'Off', 'Int', 'TklW']
prem_outfield[cols_to_change] = prem_outfield[cols_to_change].apply(lambda x: x / prem_outfield['90s'])
prem_outfield = prem_outfield.rename(columns={'Gls': 'Gls/90', 'Ast': 'Ast/90', 'Sh': 'Sh/90', 
                                              'Fls': 'Fls/90', 'Fld': 'Fld/90', 'Off': 'Off/90', 
                                              'Int': 'Int/90', 'TklW': 'TklW/90'})
prem_outfield = prem_outfield.drop(columns='90s')
prem_outfield.head()

,Gls/90,Ast/90,Sh/90,SoT/90,G/Sh,G/SoT,Fls/90,Fld/90,Off/90,Int/90,TklW/90
Rk,,,,,,,,,,,
1,0.147059,0.183824,1.727941,0.62,0.09,0.24,0.735294,1.875000,0.183824,0.625000,0.992647
25,0.101523,0.101523,0.507614,0.30,0.20,0.33,1.827411,0.659898,0.050761,1.522843,1.218274
34,0.247934,0.082645,1.652893,0.74,0.15,0.33,2.396694,1.900826,0.413223,0.495868,1.404959
40,0.000000,0.000000,0.819672,0.25,0.00,0.00,0.819672,0.409836,0.081967,1.147541,1.721311
55,0.000000,0.000000,0.681818,0.06,0.00,0.00,0.738636,0.852273,0.000000,1.477273,1.193182


In [29]:
ADJUSTED_FEATURES = ['Gls/90', 'Ast/90', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'Fls/90', 'Fld/90', 'Off/90', 'Int/90', 'TklW/90',]

scaler = StandardScaler()

prem_outfield[ADJUSTED_FEATURES] = scaler.fit_transform(prem_outfield[ADJUSTED_FEATURES])
prem_outfield.head()

,Gls/90,Ast/90,Sh/90,SoT/90,G/Sh,G/SoT,Fls/90,Fld/90,Off/90,Int/90,TklW/90
Rk,,,,,,,,,,,
1,0.165347,1.088148,0.654113,0.659619,0.145267,0.017640,-0.527613,1.492903,0.212143,-0.224866,0.138402
25,-0.142570,0.185288,-0.762689,-0.253544,1.617279,0.438107,1.561762,-0.501346,-0.491235,1.734714,0.565402
34,0.847471,-0.021811,0.566981,1.002055,0.948182,0.438107,2.650882,1.535290,1.424768,-0.506702,0.918704
40,-0.829074,-0.928444,-0.400389,-0.396226,-1.059106,-1.103605,-0.366186,-0.911754,-0.326279,0.915602,1.517403
55,-0.829074,-0.928444,-0.560438,-0.938417,-1.059106,-1.103605,-0.521219,-0.185618,-0.759564,1.635256,0.517915


Outfield players sorted, now for Goalkeepers

In [30]:
prem_keepers = prem[prem['Pos'] == 'GK']
prem_keepers = prem_keepers[GOAL_KEEPER_FEATURES]
#found no missing values

cols_to_change = ['GA', 'Saves', 'CS', 'PKsv']
prem_keepers[cols_to_change] = prem_keepers[cols_to_change].apply(lambda x : x / prem_keepers['90s'])

prem_keepers = prem_keepers.rename(columns={'GA': 'GA/90', 'Saves': 'Saves/90', 'CS':'CS/90', 'PKsv': 'PKsv/90'})
prem_keepers = prem_keepers.drop(columns='90s')

ADJUSTED_COLS = ['GA/90', 'Saves/90', 'Save%', 'CS%', 'CS/90', 'PKsv/90']
prem_keepers[ADJUSTED_COLS] = scaler.fit_transform(prem_keepers[ADJUSTED_COLS])

prem_keepers.head()

,GA/90,Saves/90,Save%,CS/90,CS%,PKsv/90
Rk,,,,,,
84,-0.722379,-1.237309,-0.573942,0.550248,0.559955,-0.606464
143,1.431302,2.132114,0.689501,-2.131615,-2.127827,-0.606464
639,-0.607882,0.169671,1.059289,-0.150693,-0.146897,-0.606464
753,-1.833669,-1.208859,1.305815,1.713703,1.720588,0.899562
773,2.016052,1.772832,-0.635574,-1.135494,-1.132999,-0.606464


Now for LaLiga

In [31]:
laliga = players[players['Comp'] == 'es La Liga']

laliga.head()

,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,Min,...,+/-90,On-Off,2CrdY,Fls,Fld,Off,Crs,Int,TklW,OG
Rk,,,,,,,,,,,,,,,,,,,,,
15,Abdel Abqar,ma MAR,DF,Getafe,es La Liga,26.0,1999.0,23,17,1437,...,0.06,0.38,0,30,36,1,2,20,17,0
23,Akor Adams,ng NGA,FW,Sevilla,es La Liga,25.0,2000.0,32,21,2064,...,-0.35,0.05,0,36,12,41,17,1,4,0
38,David Affengruber,at AUT,DF,Elche,es La Liga,24.0,2001.0,36,33,2949,...,-0.12,0.64,0,34,35,2,8,50,50,0
41,Julen Agirrezabala,es ESP,GK,Valencia,es La Liga,24.0,2000.0,18,18,1620,...,-0.72,-0.92,0,0,1,0,0,0,1,0
42,Lucien Agoume,fr FRA,MF,Sevilla,es La Liga,23.0,2002.0,34,31,2686,...,-0.30,0.31,0,58,16,8,18,49,40,0


In [32]:
laliga_outfield = laliga[OUTFIELD_FEATURES]

#missing features in G/Sh and G/SoT, fill with 0
laliga_outfield['G/Sh'] = laliga_outfield['G/Sh'].fillna(0)
laliga_outfield['G/SoT'] = laliga_outfield['G/SoT'].fillna(0)

cols_to_change = ['Gls', 'Ast', 'Sh', 'Fls', 'Fld', 'Off', 'Int', 'TklW']
laliga_outfield[cols_to_change] = laliga_outfield[cols_to_change].apply(lambda x : x / laliga_outfield['90s'])

laliga_outfield = laliga_outfield.rename(columns={'Gls': 'Gls/90', 'Ast': 'Ast/90', 'Sh': 'Sh/90', 
                                              'Fls': 'Fls/90', 'Fld': 'Fld/90', 'Off': 'Off/90', 
                                              'Int': 'Int/90', 'TklW': 'TklW/90'})
laliga_outfield = laliga_outfield.drop(columns='90s')

ADJUSTED_FEATURES = ['Gls/90', 'Ast/90', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'Fls/90', 'Fld/90', 'Off/90', 'Int/90', 'TklW/90',]

laliga_outfield[ADJUSTED_FEATURES] = scaler.fit_transform(laliga_outfield[ADJUSTED_FEATURES])

laliga_outfield.head()

,Gls/90,Ast/90,Sh/90,SoT/90,G/Sh,G/SoT,Fls/90,Fld/90,Off/90,Int/90,TklW/90
Rk,,,,,,,,,,,
15,-0.760052,0.378415,-0.769907,-0.815250,-1.042017,-1.018288,1.306971,1.563255,-0.446649,1.096545,0.286804
23,1.816479,0.441965,1.543027,2.087036,0.412767,-0.045116,0.757200,-0.837800,6.047125,-1.521552,-1.531221
38,-0.580166,-0.621892,-0.780993,-0.675940,-0.116245,0.039507,-0.214534,-0.082343,-0.452378,1.692053,1.232628
41,-0.760052,-0.944572,-1.224441,-0.954559,-1.042017,-1.018288,-2.095667,-1.489488,-0.681536,-1.616324,-1.775140
42,-0.562057,0.120921,-0.701487,-0.792031,-0.116245,1.097302,1.436378,-0.819861,0.327377,1.952283,0.859720


In [33]:
laliga_keepers = laliga[laliga['Pos'] == 'GK']
laliga_keepers = laliga_keepers[GOAL_KEEPER_FEATURES]
#found no missing values

cols_to_change = ['GA', 'Saves', 'CS', 'PKsv']
laliga_keepers[cols_to_change] = laliga_keepers[cols_to_change].apply(lambda x : x / laliga_keepers['90s'])

laliga_keepers = laliga_keepers.rename(columns={'GA': 'GA/90', 'Saves': 'Saves/90', 'CS':'CS/90', 'PKsv': 'PKsv/90'})
laliga_keepers = laliga_keepers.drop(columns='90s')

ADJUSTED_COLS = ['GA/90', 'Saves/90', 'Save%', 'CS%', 'CS/90', 'PKsv/90']
laliga_keepers[ADJUSTED_COLS] = scaler.fit_transform(laliga_keepers[ADJUSTED_COLS])

laliga_keepers.head()

,GA/90,Saves/90,Save%,CS/90,CS%,PKsv/90
Rk,,,,,,
41,1.232244,-0.089289,-1.336105,-0.142401,-0.142596,1.592642
240,-0.934690,-0.596391,0.709181,0.802792,0.807771,0.422079
601,-1.733517,-1.763701,0.888143,1.574571,1.571792,-0.894804
719,-0.328683,-0.778948,-0.185632,0.116765,0.118289,-0.894804
734,-0.073258,-0.041061,0.223425,-0.519369,-0.515288,-0.894804


Now for Ligue 1 France

In [34]:
ligue1 = players[players['Comp'] == 'fr Ligue 1']

ligue1.head()

,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,Min,...,+/-90,On-Off,2CrdY,Fls,Fld,Off,Crs,Int,TklW,OG
Rk,,,,,,,,,,,,,,,,,,,,,
6,Himad Abdelli,dz ALG,MF,Angers,fr Ligue 1,25.0,1999.0,13,11,943,...,-0.10,0.67,0,16,12,1,9,13,17,0
7,Ali Abdi,tn TUN,"MF,DF",Nice,fr Ligue 1,31.0,1993.0,22,13,1187,...,0.15,1.35,1,22,18,1,33,10,11,0
8,Salis Abdul Samed,gh GHA,MF,Nice,fr Ligue 1,25.0,2000.0,18,12,911,...,-0.20,0.68,0,19,10,0,2,11,7,0
9,Saud Abdulhamid,sa KSA,MF,Lens,fr Ligue 1,26.0,1999.0,25,14,1352,...,1.20,0.51,0,26,14,1,47,9,22,0
11,Laurent Abergel,fr FRA,MF,Lorient,fr Ligue 1,32.0,1993.0,28,27,2318,...,-0.04,0.20,0,24,27,1,9,28,23,0


In [35]:
ligue1_outfield = ligue1[ligue1['Pos'] != 'GK']
ligue1_outfield = ligue1_outfield[OUTFIELD_FEATURES]

#found missing values in G/SoT, fill with 0
ligue1_outfield['G/SoT'] = ligue1_outfield['G/SoT'].fillna(0)

cols_to_change = ['Gls', 'Ast', 'Sh', 'Fls', 'Fld', 'Off', 'Int', 'TklW', '90s']

ligue1_outfield[cols_to_change] = ligue1_outfield[cols_to_change].apply(lambda x : x / ligue1_outfield['90s'])
ligue1_outfield = ligue1_outfield.rename(columns={'Gls':'Gls/90', 'Ast':'Ast/90', 'Sh':'Sh/90', 
                                                  'Fls':'Fls/90', 'Fld':'Fld/90', 'Off':'Off/90', 
                                                  'Int':'Int/90', 'TklW':'TklW/90'})
ligue1_outfield = ligue1_outfield.drop(columns='90s')

ADJUSTED_FEATURES = ['Gls/90', 'Ast/90', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'Fls/90', 'Fld/90', 'Off/90', 'Int/90', 'TklW/90',]
ligue1_outfield[ADJUSTED_FEATURES] = scaler.fit_transform(ligue1_outfield[ADJUSTED_FEATURES])

ligue1_outfield.head()

,Gls/90,Ast/90,Sh/90,SoT/90,G/Sh,G/SoT,Fls/90,Fld/90,Off/90,Int/90,TklW/90
Rk,,,,,,,,,,,
6,0.249575,-1.005723,-0.226768,0.078047,-1.114796,-1.104691,0.693084,0.048889,-0.317420,0.810676,1.414412
7,0.457879,-1.005723,-0.046899,-0.350606,1.407235,2.305288,0.995567,0.430487,-0.397646,-0.230368,-0.370932
8,-0.828707,-1.005723,-1.263987,-1.065029,-1.114796,-1.104691,1.449792,-0.215140,-0.709633,0.487898,-0.689648
9,-0.073910,1.742112,-0.789093,-0.588747,2.203666,1.941557,1.136726,-0.313255,-0.435084,-0.571756,1.068164
11,-0.609289,-1.005723,-0.439880,-0.612561,-0.451103,-0.195363,-0.563747,-0.117636,-0.550011,0.479584,-0.238824


In [36]:
ligue1_keepers = ligue1[ligue1['Pos'] == 'GK']
ligue1_keepers = ligue1_keepers[GOAL_KEEPER_FEATURES]
#no missing values

cols_to_change = ['GA', 'Saves', 'CS', 'PKsv']
ligue1_keepers[cols_to_change] = ligue1_keepers[cols_to_change].apply(lambda x : x / ligue1_keepers['90s'])

ligue1_keepers = ligue1_keepers.rename(columns={'GA': 'GA/90', 'Saves': 'Saves/90', 'CS':'CS/90', 'PKsv': 'PKsv/90'})
ligue1_keepers = ligue1_keepers.drop(columns='90s')

ADJUSTED_COLS = ['GA/90', 'Saves/90', 'Save%', 'CS%', 'CS/90', 'PKsv/90']
ligue1_keepers[ADJUSTED_COLS] = scaler.fit_transform(ligue1_keepers[ADJUSTED_COLS])

ligue1_keepers.head()

,GA/90,Saves/90,Save%,CS/90,CS%,PKsv/90
Rk,,,,,,
533,-2.047195,-1.772937,0.765331,2.170765,2.185259,0.633133
591,0.852594,0.604304,-0.429523,-0.388234,-0.341069,0.043938
704,0.025352,-0.172851,0.039048,-0.465981,-0.495003,0.929485
731,1.155706,1.347424,0.085905,-0.871980,-0.866256,-0.039358
890,2.503870,1.369511,-1.015236,-0.885230,-0.866256,-0.049054


Now for Bundesliga

In [37]:
bundesliga = players[players['Comp'] == 'de Bundesliga']
bundesliga.head()

,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,Min,...,+/-90,On-Off,2CrdY,Fls,Fld,Off,Crs,Int,TklW,OG
Rk,,,,,,,,,,,,,,,,,,,,,
21,Ragnar Ache,de GER,FW,Köln,de Bundesliga,27.0,1998.0,29,18,1718,...,-0.58,-0.38,0,18,29,10,1,4,8,0
29,Karim Adeyemi,de GER,"MF,FW",Dortmund,de Bundesliga,23.0,2002.0,28,15,1195,...,0.98,-0.13,0,21,25,5,11,8,4,0
102,Aurele Amenda,ch SUI,DF,Eintracht Frankfurt,de Bundesliga,22.0,2003.0,24,18,1678,...,-0.32,-0.45,0,22,13,0,6,29,12,1
106,Nadiem Amiri,de GER,MF,Mainz 05,de Bundesliga,28.0,1996.0,26,24,2083,...,-0.04,0.69,0,20,26,3,136,19,18,0
110,Mohamed Amoura,dz ALG,"FW,MF",Wolfsburg,de Bundesliga,25.0,2000.0,30,23,1919,...,0.00,1.89,0,21,11,12,34,7,22,0


In [38]:
bundes_outfield = bundesliga[bundesliga['Pos'] != 'GK']
bundes_outfield = bundes_outfield[OUTFIELD_FEATURES]

#found missing values in G/SoT, G/Sh, fill with 0
bundes_outfield['G/SoT'] = bundes_outfield['G/SoT'].fillna(0)
bundes_outfield['G/Sh'] = bundes_outfield['G/Sh'].fillna(0)

cols_to_change = ['Gls', 'Ast', 'Sh', 'Fls', 'Fld', 'Off', 'Int', 'TklW', '90s']

bundes_outfield[cols_to_change] = bundes_outfield[cols_to_change].apply(lambda x : x / bundes_outfield['90s'])
bundes_outfield = bundes_outfield.rename(columns={'Gls':'Gls/90', 'Ast':'Ast/90', 'Sh':'Sh/90', 
                                                  'Fls':'Fls/90', 'Fld':'Fld/90', 'Off':'Off/90', 
                                                  'Int':'Int/90', 'TklW':'TklW/90'})
bundes_outfield = bundes_outfield.drop(columns='90s')

ADJUSTED_FEATURES = ['Gls/90', 'Ast/90', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'Fls/90', 'Fld/90', 'Off/90', 'Int/90', 'TklW/90',]
bundes_outfield[ADJUSTED_FEATURES] = scaler.fit_transform(bundes_outfield[ADJUSTED_FEATURES])

bundes_outfield.head()

,Gls/90,Ast/90,Sh/90,SoT/90,G/Sh,G/SoT,Fls/90,Fld/90,Off/90,Int/90,TklW/90
Rk,,,,,,,,,,,
21,1.073727,0.833874,1.481079,1.527970,0.434875,0.170729,-0.180490,1.031348,1.542319,-1.372400,-1.247967
29,1.927786,1.630204,1.061152,1.296218,1.814259,0.992318,1.194357,1.705811,0.884902,-0.566522,-1.532578
102,-0.884720,-0.992193,-0.861335,-0.882249,-1.358325,-1.289874,0.338717,-0.497964,-0.789320,1.401794,-0.702556
106,1.891260,-0.237261,1.068480,1.064466,-0.392756,-0.468285,-0.345953,0.298265,-0.210953,-0.112266,-0.379474
110,1.122327,0.235901,1.582273,0.925415,0.296936,0.398948,-0.086520,-0.838565,1.719655,-1.127367,0.231802


In [45]:
bundes_keepers = bundesliga[bundesliga['Pos'] == 'GK']
bundes_keepers = bundes_keepers[GOAL_KEEPER_FEATURES]
#no missing values

cols_to_change = ['GA', 'Saves', 'CS', 'PKsv']
bundes_keepers[cols_to_change] = bundes_keepers[cols_to_change].apply(lambda x : x / bundes_keepers['90s'])

bundes_keepers = bundes_keepers.rename(columns={'GA': 'GA/90', 'Saves': 'Saves/90', 'CS':'CS/90', 'PKsv': 'PKsv/90'})
bundes_keepers = bundes_keepers.drop(columns='90s')

ADJUSTED_COLS = ['GA/90', 'Saves/90', 'Save%', 'CS%', 'CS/90', 'PKsv/90']
bundes_keepers[ADJUSTED_COLS] = scaler.fit_transform(bundes_keepers[ADJUSTED_COLS])

bundes_keepers.head()

,GA/90,Saves/90,Save%,CS/90,CS%,PKsv/90
Rk,,,,,,
171,0.231125,-0.119309,-0.646549,-0.440059,-0.438738,0.316065
186,0.414159,0.218012,-0.183838,-0.608555,-0.598411,-0.672563
243,-0.741570,0.600310,0.964371,-0.323351,-0.326967,-0.672563
245,-0.134943,-0.227252,0.056086,-0.194972,-0.199228,-0.672563
325,-1.766966,1.788031,2.523879,0.516492,0.551235,-0.672563


Finally Serie A

In [46]:
serieA = players[players['Comp'] == 'it Serie A']
serieA.head()

,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,Min,...,+/-90,On-Off,2CrdY,Fls,Fld,Off,Crs,Int,TklW,OG
Rk,,,,,,,,,,,,,,,,,,,,,
20,Francesco Acerbi,it ITA,DF,Inter,it Serie A,37.0,1988.0,18,15,1378,...,1.04,-0.63,0,17,13,3,3,18,9,0
24,Che Adams,sct SCO,FW,Torino,it Serie A,29.0,1996.0,33,19,1893,...,-0.19,0.69,0,18,15,9,14,1,9,0
37,Michel Aebischer,ch SUI,MF,Pisa,it Serie A,28.0,1997.0,35,33,2892,...,-0.81,2.43,0,44,38,0,43,35,37,0
52,Honest Ahanor,it ITA,DF,Atalanta,it Serie A,17.0,2008.0,22,15,1377,...,0.13,-0.44,0,24,28,0,2,24,20,0
62,Manuel Akanji,ch SUI,DF,Inter,it Serie A,30.0,1995.0,33,31,2821,...,1.50,0.45,0,26,18,2,3,37,24,0


In [47]:
serieA_outfield = serieA[serieA['Pos'] != 'GK']
serieA_outfield = serieA_outfield[OUTFIELD_FEATURES]

#Missing values in G/Sh and G/SoT, fill with 0
serieA_outfield['G/SoT'] = serieA_outfield['G/SoT'].fillna(0)
serieA_outfield['G/Sh'] = serieA_outfield['G/Sh'].fillna(0)

cols_to_change = ['Gls', 'Ast', 'Sh', 'Fls', 'Fld', 'Off', 'Int', 'TklW', '90s']

serieA_outfield[cols_to_change] = serieA_outfield[cols_to_change].apply(lambda x : x / serieA_outfield['90s'])
serieA_outfield = serieA_outfield.rename(columns={'Gls':'Gls/90', 'Ast':'Ast/90', 'Sh':'Sh/90', 
                                                  'Fls':'Fls/90', 'Fld':'Fld/90', 'Off':'Off/90', 
                                                  'Int':'Int/90', 'TklW':'TklW/90'})
serieA_outfield = serieA_outfield.drop(columns='90s')

ADJUSTED_FEATURES = ['Gls/90', 'Ast/90', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'Fls/90', 'Fld/90', 'Off/90', 'Int/90', 'TklW/90',]
serieA_outfield[ADJUSTED_FEATURES] = scaler.fit_transform(serieA_outfield[ADJUSTED_FEATURES])

serieA_outfield.head()

,Gls/90,Ast/90,Sh/90,SoT/90,G/Sh,G/SoT,Fls/90,Fld/90,Off/90,Int/90,TklW/90
Rk,,,,,,,,,,,
20,-0.891301,-0.201980,-1.135813,-0.920963,-1.158428,-1.161492,-0.346170,-0.450448,0.150791,0.950170,-0.720037
24,1.116126,0.137833,1.375142,1.183835,0.364941,0.295370,-0.846473,-0.630571,1.163865,-1.580701,-1.098518
37,-0.672423,-0.236716,-0.708835,-0.681176,-0.465987,-0.278545,0.165237,-0.005917,-0.703609,0.757072,0.617895
52,-0.891301,-0.945322,-0.430772,-0.414746,-1.158428,-1.161492,0.555110,0.853889,-0.703609,1.829381,0.984234
62,-0.442356,-0.945322,-0.554661,-0.601247,0.087965,0.295370,-0.898621,-0.815774,-0.425178,0.962811,-0.296815


In [48]:
serieA_keepers = serieA[serieA['Pos'] == 'GK']
serieA_keepers = serieA_keepers[GOAL_KEEPER_FEATURES]
#no missing values

cols_to_change = ['GA', 'Saves', 'CS', 'PKsv']
serieA_keepers[cols_to_change] = serieA_keepers[cols_to_change].apply(lambda x : x / serieA_keepers['90s'])

serieA_keepers = serieA_keepers.rename(columns={'GA': 'GA/90', 'Saves': 'Saves/90', 'CS':'CS/90', 'PKsv': 'PKsv/90'})
serieA_keepers = serieA_keepers.drop(columns='90s')

ADJUSTED_COLS = ['GA/90', 'Saves/90', 'Save%', 'CS%', 'CS/90', 'PKsv/90']
serieA_keepers[ADJUSTED_COLS] = scaler.fit_transform(serieA_keepers[ADJUSTED_COLS])

serieA_keepers.head()

,GA/90,Saves/90,Save%,CS/90,CS%,PKsv/90
Rk,,,,,,
173,0.922794,1.590781,0.156488,-0.021458,0.002606,1.059094
314,0.284782,0.243966,0.722108,-0.029862,-0.095684,-0.836454
414,-1.581008,-0.962680,1.446102,1.541599,1.575245,0.011554
461,0.654334,0.773673,-0.296008,-1.022363,-1.007099,-0.836454
480,-0.934068,0.477354,1.197229,0.224970,0.243863,0.905401


In [50]:
outfield_vectors = pd.concat([prem_outfield, laliga_outfield, bundes_outfield, ligue1_outfield, serieA_outfield])
gk_vectors = pd.concat([prem_keepers, laliga_keepers, bundes_keepers, ligue1_keepers, serieA_keepers])

outfield_vectors.head()

,Gls/90,Ast/90,Sh/90,SoT/90,G/Sh,G/SoT,Fls/90,Fld/90,Off/90,Int/90,TklW/90
Rk,,,,,,,,,,,
1,0.165347,1.088148,0.654113,0.659619,0.145267,0.017640,-0.527613,1.492903,0.212143,-0.224866,0.138402
25,-0.142570,0.185288,-0.762689,-0.253544,1.617279,0.438107,1.561762,-0.501346,-0.491235,1.734714,0.565402
34,0.847471,-0.021811,0.566981,1.002055,0.948182,0.438107,2.650882,1.535290,1.424768,-0.506702,0.918704
40,-0.829074,-0.928444,-0.400389,-0.396226,-1.059106,-1.103605,-0.366186,-0.911754,-0.326279,0.915602,1.517403
55,-0.829074,-0.928444,-0.560438,-0.938417,-1.059106,-1.103605,-0.521219,-0.185618,-0.759564,1.635256,0.517915


Save the cleaned and processed data as CSV files for future use

In [52]:
outfield_vectors.to_csv("../data/clean/outfield_vectors.csv")
gk_vectors.to_csv("../data/clean/gk_vectors.csv")
players.to_csv("../data/clean/players.csv")

In [55]:
haaland = players[players['Player'] == 'Erling Haaland']
haaland_index = 1105

Player        Erling Haaland
Nation                no NOR
Pos                       FW
Squad        Manchester City
Comp      eng Premier League
                 ...        
Off                        5
Crs                        3
Int                        4
TklW                       7
OG                         0
Name: 1105, Length: 62, dtype: object